In [1]:
# Posts Scraping
import requests
import csv
import time

def download_posts(subreddit, title_filter=None, output_file=None):
    base_url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    all_posts = []
    before = "2026-06-01T00:00:00"
    print(f"Downloading r/{subreddit}...")

    while True:
        params = {
            "subreddit": subreddit,
            "before": before,
            "after": "2023-06-01T00:00:00",
            "limit": 100,
            "sort": "desc"
        }
        if title_filter:
            params["title"] = title_filter
        response = requests.get(base_url, params=params)
        data = response.json().get("data") or []
        if not data:
            print("No more data. ")
            break
        all_posts.extend(data)
        print(f"{len(all_posts)} posts downloaded. ")
        before = data[-1].get("created_utc") or data[-1].get("created")
        if len(data) < 100:
            break
        time.sleep(1)

    if all_posts and output_file:
        all_keys = set()
        for post in all_posts:
            all_keys.update(post.keys())
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=all_keys, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(all_posts)
        print(f"Saved {len(all_posts)} posts to {output_file}")

download_posts("aphantasia", output_file="Aphantasia Posts.csv")
other_subreddits = ["Meditation"]
for sub in other_subreddits:
    download_posts(sub, title_filter="aphantasia", output_file=f"{sub} Posts.csv")

C:\Users\SP_Hy\anaconda3\envs\mamba_env\envs\Thesis\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


100 posts downloaded.
200 posts downloaded.
300 posts downloaded.
400 posts downloaded.
500 posts downloaded.
600 posts downloaded.
700 posts downloaded.
800 posts downloaded.
900 posts downloaded.
1000 posts downloaded.
1100 posts downloaded.
1200 posts downloaded.
1300 posts downloaded.
1400 posts downloaded.
1500 posts downloaded.
1600 posts downloaded.
1700 posts downloaded.
1800 posts downloaded.
1900 posts downloaded.
2000 posts downloaded.
2100 posts downloaded.
2200 posts downloaded.
2300 posts downloaded.
2400 posts downloaded.
2500 posts downloaded.
2600 posts downloaded.
2700 posts downloaded.
2800 posts downloaded.
2900 posts downloaded.
3000 posts downloaded.
3100 posts downloaded.
3200 posts downloaded.
3300 posts downloaded.
3400 posts downloaded.
3500 posts downloaded.
3600 posts downloaded.
3700 posts downloaded.
3800 posts downloaded.
3900 posts downloaded.
4000 posts downloaded.
4100 posts downloaded.
4200 posts downloaded.
4300 posts downloaded.
4400 posts downloade

In [9]:
# Primary Posts Cleaning
import pandas as pd
import re

keywords = r'meditat\w*'

def clean_posts(filepath):
    df = pd.read_csv(filepath, low_memory=False)
    keep_cols = ['id', 'title', 'selftext', 'author', 'score', 'num_comments', 'subreddit']
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]
    df['selftext'] = df['selftext'].astype(str)
    df['selftext'] = df['selftext'].apply(lambda x: re.sub(r'\n+', ' ', x).strip())
    df = df[~df['selftext'].isin(['[deleted]', '[removed]', 'nan', ''])]
    df = df[df['selftext'].str.strip() != '']
    mask = (
        df['title'].str.contains(keywords, case=False, na=False)
    )
    df = df[mask]
    return df.reset_index(drop=True)

input_files = ["Aphantasia Posts.csv", "Meditation Posts.csv"]

all_dfs = []
for f in input_files:
    cleaned = clean_posts(f)
    print(f"{f}: {len(cleaned)} posts left. ")
    all_dfs.append(cleaned)

combined = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='id')
combined.to_csv("Meditation Posts Cleaned.csv", index=False, encoding="utf-8")
print(f"Finished. {len(combined)} posts in total. ")

Aphantasia Posts.csv: 68 posts left. 
Meditation Posts.csv: 15 posts left. 
Finished. 83 posts in total. 


In [151]:
# Primary Comments Scraping (Based on post_id)
import requests
import csv
import time
import pandas as pd
import threading

def flatten_comments(data, result=None):
    if result is None:
        result = []
    for item in data:
        if isinstance(item, dict):
            kind = item.get('kind')
            if kind == 't1':
                comment_data = item.get('data', {})
                result.append({
                    'id': comment_data.get('id'),
                    'body': comment_data.get('body'),
                    'author': comment_data.get('author'),
                    'score': comment_data.get('score'),
                    'link_id': comment_data.get('link_id'),
                })
                replies = comment_data.get('replies')
                if isinstance(replies, dict):
                    children = replies.get('data', {}).get('children', [])
                    flatten_comments(children, result)
    return result

def download_comments_by_post_id(post_id, timeout=4, retries=1):
    for attempt in range(retries):
        result = []
        def fetch():
            try:
                response = requests.get(
                    "https://arctic-shift.photon-reddit.com/api/comments/tree",
                    params={"link_id": f"t3_{post_id}", "limit": 9999},
                    timeout=3
                )
                data = response.json().get("data") or []
                result.extend(flatten_comments(data))
            except:
                pass
        t = threading.Thread(target=fetch, daemon=True)
        t.start()
        t.join(timeout=timeout)
        if not t.is_alive():
            return result
        print(f"Retry {attempt+1} on {post_id}")
    print(f"Skipping {post_id}")
    return []

def download_all_comments(posts_csv, output_file="Meditation Comments.csv"):
    df = pd.read_csv(posts_csv)
    post_ids = df['id'].tolist()
    all_comments = []
    print(f"Start downloading comments for {len(post_ids)} posts. ")
    for i, post_id in enumerate(post_ids):
        comments = download_comments_by_post_id(post_id)
        all_comments.extend(comments)
        print(f"[{i+1}/{len(post_ids)}] Post {post_id}: {len(comments)} comments, total {len(all_comments)}")
        time.sleep(0.1)
    if all_comments:
        fieldnames = ['id', 'body', 'author', 'score', 'link_id']
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(all_comments)
        print(f"Finished. {len(all_comments)} comments saved to {output_file}. ")

if __name__ == "__main__":
    download_all_comments("Meditation Posts Cleaned.csv")

Start downloading comments for 83 posts. 
[1/83] Post 1sp19mb: 0 comments, total 0
[2/83] Post 1sbzhvb: 8 comments, total 8
[3/83] Post 1sbqpij: 19 comments, total 27
[4/83] Post 1rm9977: 8 comments, total 35
[5/83] Post 1qp73ex: 49 comments, total 84
[6/83] Post 1qp0gb2: 9 comments, total 93
[7/83] Post 1qg81sb: 7 comments, total 100
[8/83] Post 1qfpifl: 25 comments, total 125
[9/83] Post 1qfo57w: 11 comments, total 136
[10/83] Post 1qaim5f: 2 comments, total 138
[11/83] Post 1pqug8h: 0 comments, total 138
[12/83] Post 1nawcb1: 0 comments, total 138
[13/83] Post 1n8kvdf: 0 comments, total 138
[14/83] Post 1n1elen: 26 comments, total 164
[15/83] Post 1lrf49h: 47 comments, total 211
[16/83] Post 1ldbflm: 11 comments, total 222
[17/83] Post 1kiec0e: 31 comments, total 253
[18/83] Post 1k8btea: 11 comments, total 264
[19/83] Post 1jxgpeg: 4 comments, total 268
[20/83] Post 1jn2bj2: 32 comments, total 300
[21/83] Post 1j1d448: 33 comments, total 333
[22/83] Post 1iqzxm3: 7 comments, total 

In [11]:
# Primary Posts and Comments Hashed
import pandas as pd
import hashlib

def hash_id(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

# Posts
posts = pd.read_csv("Meditation Posts Cleaned.csv")
posts['id'] = posts['id'].apply(hash_id)
posts['author'] = posts['author'].apply(hash_id)
posts.to_csv("Meditation Posts Hashed.csv", index=False, encoding="utf-8")
print(f"Posts done: {len(posts)}")

# Comments
comments = pd.read_csv("Meditation Comments.csv", lineterminator='\n')
comments.columns = comments.columns.str.strip()
comments['id'] = comments['id'].apply(hash_id)
comments['author'] = comments['author'].apply(hash_id)
comments['link_id'] = comments['link_id'].str.replace('t3_', '', regex=False)
comments['link_id'] = comments['link_id'].apply(hash_id)
comments['body'] = comments['body'].apply(lambda x: str(x).replace('\n', ' ').replace('\r', ' '))
comments.to_csv("Meditation Comments Hashed.csv", index=False, encoding="utf-8")
print(f"Comments done: {len(comments)}")

Posts done: 83
Comments done: 1499


In [1]:
# Primary Comments Cleaning
import pandas as pd
import re

df = pd.read_csv("Meditation Comments Hashed.csv", lineterminator='\n')
df.columns = df.columns.str.strip()

df['body'] = df['body'].astype(str)
df['body'] = df['body'].apply(lambda x: re.sub(r'\n+', ' ', x).strip())
df = df[~df['body'].isin(['[deleted]', '[removed]', 'nan', ''])]
df = df[df['body'].str.strip() != '']
df = df[df['body'].str.len() > 20]
keep_cols = ['id', 'body', 'author', 'score', 'link_id']
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]
df = df.reset_index(drop=True)
df.to_csv("Meditation Comments Cleaned.csv", index=False, encoding="utf-8")
print(f"Finished. {len(df)} comments left. ")

Finished. 1420 comments left. 


In [8]:
# Transform CSV to Word File (Primary Posts)
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

df = pd.read_csv("Meditation Posts Hashed.csv")
df['text_len'] = df['selftext'].str.len()
df = df.sort_values('text_len', ascending=False)
df = df.drop_duplicates(subset='selftext', keep='first')
df = df.drop(columns='text_len')
df = df.sort_values('title', key=lambda x: x.str.lower())
df = df.reset_index(drop=True)
df.to_csv("Posts_Meditation_Title.csv", index=False, encoding="utf-8")
print(f"Total {len(df)} records. ")

df = pd.read_csv("Posts_Meditation_Title.csv")
mask = df['title'].str.contains(r'meditat\w*', case=False, na=False)
df = df[mask].reset_index(drop=True)
doc = Document()
title = doc.add_heading('Aphantasia & Meditation – Posts for Analysis', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

for i, row in df.iterrows():
    heading = doc.add_paragraph()
    run = heading.add_run(f"{i+1}. {row['title']}")
    run.bold = True
    run.font.size = Pt(14)
    meta = doc.add_paragraph()
    meta_run = meta.add_run(
        f"Subreddit: {row['subreddit']} | Score: {row['score']} | Comments: {row['num_comments']}"
    )
    meta_run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)
    meta_run.font.size = Pt(9)
    doc.add_paragraph(str(row['selftext']))
    doc.add_paragraph("")
doc.save("Posts_Meditation.docx")
print(f"Done! {len(df)} posts saved. ")

Total 78 records. 
Done! 78 posts saved. 


In [2]:
# Comments Filtering (Primary Posts)
import pandas as pd

posts = pd.read_csv("Posts_Meditation_Title.csv")
comments = pd.read_csv("Meditation Comments Cleaned.csv")
comments['link_id_clean'] = comments['link_id'].str.replace('t3_', '', regex=False)
filtered_comments = comments[comments['link_id_clean'].isin(posts['id'])]
filtered_comments['text_len'] = filtered_comments['body'].str.len()
filtered_comments = filtered_comments.sort_values('text_len', ascending=False)
filtered_comments = filtered_comments.drop_duplicates(subset='body', keep='first')
filtered_comments = filtered_comments.drop(columns=['text_len', 'link_id_clean'])
# Remove Unsupported Posts
filtered_comments = filtered_comments[filtered_comments['score'] > 0]
filtered_comments.to_csv("Comments_Meditation_Title.csv", index=False, encoding="utf-8")
print(f"Total {len(filtered_comments)} comments. ")

Total 1366 comments. 


C:\Users\SP_Hy\AppData\Local\Temp\ipykernel_30804\3548921586.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_comments['text_len'] = filtered_comments['body'].str.len()


In [13]:
# Primary Posts
# Transform Comments CSV to Word File (Group by Posts)
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

posts = pd.read_csv("Posts_Meditation_Title.csv")
comments = pd.read_csv("Comments_Meditation_Title.csv")
comments['link_id_clean'] = comments['link_id'].str.replace('t3_', '', regex=False)
doc = Document()
title = doc.add_heading('Aphantasia & Meditation – Comments for Analysis', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

total_comments = 0
for i, prow in posts.iterrows():
    post_comments = comments[comments['link_id_clean'] == prow['id']]
    if len(post_comments) == 0:
        continue
    post_comments = post_comments.sort_values('score', ascending=False)
    heading = doc.add_paragraph()
    run = heading.add_run(f"{i+1}. {prow['title']}")
    run.bold = True
    run.font.size = Pt(14)
    for j, crow in enumerate(post_comments.itertuples(index=False), start=1):
        c_para = doc.add_paragraph()
        c_run = c_para.add_run(f"  [{j}] Score: {crow.score} ")
        c_run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)
        c_run.font.size = Pt(9)
        doc.add_paragraph(str(crow.body))
        total_comments += 1
    doc.add_paragraph("")
doc.save("Comments_Meditation.docx")
print(f"Done! {total_comments} comments saved under {len(posts)} posts. ")

Done! 1366 comments saved under 78 posts. 


In [3]:
# Most Frequent Words Selection
import pandas as pd
from collections import Counter
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

files = ["Aphantasia Posts.csv", "Meditation Posts.csv"]
df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
all_text = ' '.join(df['title'].astype(str) + ' ' + df['selftext'].astype(str))
all_text = all_text.lower()
words = re.findall(r'\b[a-z]{5,}\b', all_text)
stop_words = set(stopwords.words('english'))
filtered = [w for w in words if w not in stop_words]
counter = Counter(filtered)
print("Top 200 most frequent words: ")
for word, count in counter.most_common(200):
    print(f"{word}: {count}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SP_Hy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Top 200 most frequent words: 
aphantasia: 10416
people: 5122
think: 3770
things: 3712
something: 3023
would: 2722
visualize: 2641
imagine: 2639
really: 2567
could: 2340
anyone: 2281
visual: 2099
experience: 2071
remember: 1980
images: 1943
anything: 1887
memory: 1887
actually: 1788
thought: 1773
never: 1764
always: 1732
mental: 1603
brain: 1598
picture: 1598
image: 1548
thing: 1466
still: 1425
someone: 1339
thinking: 1337
without: 1326
apple: 1304
imagery: 1301
seeing: 1233
different: 1183
years: 1163
reading: 1137
https: 1102
inner: 1099
trying: 1096
dream: 1074
found: 1063
sense: 1053
imagination: 1047
since: 1044
close: 1030
dreams: 1022
thoughts: 1015
understand: 1004
maybe: 996
might: 992
first: 974
visualization: 931
memories: 913
ability: 908
sometimes: 898
others: 865
nothing: 842
words: 818
describe: 816
everyone: 809
experiences: 797
recently: 793
question: 787
person: 779
black: 775
aphants: 768
around: 767
makes: 752
vivid: 749
aphant: 742
monologue: 721
going: 719
cannot: 

In [6]:
# Supplementary Posts
import pandas as pd
import re
import hashlib

keywords = r"(inner\w*|monologue\w*|voice\w*|internal\w*|sound\w*|sense\w*|sensory\w*)"
keyword_patterns = {
    "inner": r"inner\w*",
    "monologue": r"monologue\w*",
    "voice": r"voice\w*",
    "internal": r"internal\w*",
    "sound": r"sound\w*",
    "sensory": r"sensory\w*",
    "sense": r"sense\w*"
}

def hash_id(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

def clean_posts(filepath):
    df=pd.read_csv(filepath,low_memory=False)
    keep_cols = ['id', 'title', 'selftext', 'author', 'score', 'num_comments', 'subreddit']
    keep_cols = [c for c in keep_cols if c in df.columns]
    df=df[keep_cols]
    df['selftext'] = df['selftext'].astype(str)
    df['selftext'] = df['selftext'].apply(lambda x:re.sub(r'\n+',' ',x).strip())
    df = df[~df['selftext'].isin(['[deleted]','[removed]','nan',''])]
    df = df[df['selftext'].str.strip()!='']
    return df.reset_index(drop=True)

input_files = ["Aphantasia Posts.csv", "Meditation Posts.csv"]
all_dfs = []
for f in input_files:
    all_dfs.append(clean_posts(f))
df = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='id')

# Keywords-based Posts Selection + Content-based Meditation Check
keyword_based_posts = df[(df['title'].str.contains(keywords, case=False, na=False, regex=True) | df['selftext'].str.contains(keywords, case=False, na=False, regex=True)) & (df['selftext'].str.contains(r"meditat\w*", case=False, na=False, regex=True))].copy()
print(f"Posts: {len(keyword_based_posts)}")
keyword_based_posts['hashed_id'] = keyword_based_posts['id'].apply(hash_id)

reference = pd.read_csv("Posts_Meditation_Title.csv")
reference_ids = set(reference['id'].astype(str))
all_ids = set(keyword_based_posts['hashed_id'].astype(str))
matched_ids = all_ids & reference_ids
missing_ids = all_ids - reference_ids
print(f"Matched IDs: {len(matched_ids)}")
print(f"Missing IDs: {len(missing_ids)}")

missing = keyword_based_posts[keyword_based_posts['hashed_id'].isin(missing_ids)].copy()
missing['text'] = missing['title'] + " " + missing['selftext']

results = []
total_set = set()

for word in ["inner", "monologue", "voice", "internal", "sound", "sensory", "sense"]:
    pattern = keyword_patterns[word]
    mask = missing['text'].str.contains(pattern, case=False, na=False, regex=True)
    matched = set(missing.loc[mask, 'hashed_id'])
    results.append({
        'Word': word,
        'Post Count': len(matched)
    })
result_df = pd.DataFrame(results)

print("\nWord | Post Count")
result_df = result_df.sort_values("Post Count", ascending=False)
for _, row in result_df.iterrows():
    print(f"{row['Word']} | {row['Post Count']}")
missing.to_csv("Supplementary Meditation Posts Cleaned.csv", index=False, encoding="utf-8")
print(f"Done! {len(missing)} posts saved. ")

# Supplementary Posts Hashed Saved
posts = pd.read_csv("Supplementary Meditation Posts Cleaned.csv", encoding_errors='ignore')
posts['id'] = posts['id'].apply(hash_id)
posts['author'] = posts['author'].apply(hash_id)
posts.to_csv("Supplementary Meditation Posts Hashed.csv", index=False, encoding="utf-8")
print(f"Posts done: {len(posts)}")

C:\Users\SP_Hy\AppData\Local\Temp\ipykernel_30804\1886123992.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  keyword_based_posts = df[(df["title"].str.contains(keywords, case=False, na=False, regex=True) | df["selftext"].str.contains(keywords, case=False, na=False, regex=True)) & (df["selftext"].str.contains(r"meditat\w*", case=False, na=False, regex=True))].copy()


Posts: 122
Matched IDs: 21
Missing IDs: 101

Word | Post Count
sense | 51
inner | 40
sound | 31
internal | 30
monologue | 22
voice | 21
sensory | 16
Done! 101 posts saved. 
Posts done: 101


In [1]:
# Supplementary Comments Scraping (Based on post_id)
import requests
import csv
import time
import pandas as pd
import threading

def flatten_comments(data, result=None):
    if result is None:
        result = []
    for item in data:
        if isinstance(item, dict):
            kind = item.get('kind')
            if kind == 't1':
                comment_data = item.get('data', {})
                result.append({
                    'id': comment_data.get('id'),
                    'body': comment_data.get('body'),
                    'author': comment_data.get('author'),
                    'score': comment_data.get('score'),
                    'link_id': comment_data.get('link_id'),
                })
                replies = comment_data.get('replies')
                if isinstance(replies, dict):
                    children = replies.get('data', {}).get('children', [])
                    flatten_comments(children, result)
    return result

def download_comments_by_post_id(post_id, timeout=4, retries=1):
    for attempt in range(retries):
        result = []
        def fetch():
            try:
                response = requests.get(
                    "https://arctic-shift.photon-reddit.com/api/comments/tree",
                    params={"link_id": f"t3_{post_id}", "limit": 9999},
                    timeout=3
                )
                data = response.json().get("data") or []
                result.extend(flatten_comments(data))
            except:
                pass
        t = threading.Thread(target=fetch, daemon=True)
        t.start()
        t.join(timeout=timeout)
        if not t.is_alive():
            return result
        print(f"Retry {attempt+1} on {post_id}")
    print(f"Skipping {post_id}")
    return []

def download_all_comments(posts_csv, output_file="Supplementary Meditation Comments.csv"):
    df = pd.read_csv(posts_csv, encoding_errors='ignore')
    post_ids = df['id'].tolist()
    all_comments = []
    print(f"Start downloading comments for {len(post_ids)} posts. ")
    for i, post_id in enumerate(post_ids):
        comments = download_comments_by_post_id(post_id)
        all_comments.extend(comments)
        print(f"[{i+1}/{len(post_ids)}] Post {post_id}: {len(comments)} comments, total {len(all_comments)}")
        time.sleep(0.1)
    if all_comments:
        fieldnames = ['id', 'body', 'author', 'score', 'link_id']
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(all_comments)
        print(f"Finished. {len(all_comments)} comments saved to {output_file}. ")

if __name__ == "__main__":
    download_all_comments("Supplementary Meditation Posts Cleaned.csv") # Manual Selection

Start downloading comments for 16 posts. 
[1/16] Post 1s8puko: 26 comments, total 26
[2/16] Post 1admdu9: 27 comments, total 53
[3/16] Post 1mzy89v: 16 comments, total 69
[4/16] Post 1psc8fy: 6 comments, total 75
[5/16] Post 18npbny: 5 comments, total 80
[6/16] Post 157ts4z: 7 comments, total 87
[7/16] Post 1b1ynyy: 8 comments, total 95
[8/16] Post 1eccgow: 2 comments, total 97
[9/16] Post 1p3kld1: 16 comments, total 113
[10/16] Post 1q6oeri: 30 comments, total 143
[11/16] Post 15hlhl1: 20 comments, total 163
[12/16] Post 1mq424f: 8 comments, total 171
[13/16] Post 1igbfz8: 9 comments, total 180
[14/16] Post 1ruszu5: 8 comments, total 188
[15/16] Post 1k8eule: 14 comments, total 202
[16/16] Post 1t9sywg: 31 comments, total 233
Finished. 233 comments saved to Supplementary Meditation Comments.csv. 


In [17]:
# Supplementary Comments Hashed
import pandas as pd
import hashlib

def hash_id(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

comments = pd.read_csv("Supplementary Meditation Comments.csv", lineterminator='\n')
comments.columns = comments.columns.str.strip()
comments['id'] = comments['id'].apply(hash_id)
comments['author'] = comments['author'].apply(hash_id)
comments['link_id'] = comments['link_id'].str.replace('t3_', '', regex=False)
comments['link_id'] = comments['link_id'].apply(hash_id)
comments['body'] = comments['body'].apply(lambda x: str(x).replace('\n', ' ').replace('\r', ' '))
comments.to_csv("Supplementary Meditation Comments Hashed.csv", index=False, encoding="utf-8")
print(f"Comments done: {len(comments)}")

Comments done: 233


In [18]:
# Supplementary Comments Cleaning
import pandas as pd
import re

df = pd.read_csv("Supplementary Meditation Comments Hashed.csv", lineterminator='\n')
df.columns = df.columns.str.strip()

df['body'] = df['body'].astype(str)
df['body'] = df['body'].apply(lambda x: re.sub(r'\n+', ' ', x).strip())
df = df[~df['body'].isin(['[deleted]', '[removed]', 'nan', ''])]
df = df[df['body'].str.strip() != '']
df = df[df['body'].str.len() > 20]
keep_cols = ['id', 'body', 'author', 'score', 'link_id']
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]
df = df.reset_index(drop=True)
df.to_csv("Supplementary Meditation Comments Cleaned.csv", index=False, encoding="utf-8")
print(f"Finished. {len(df)} comments left. ")

Finished. 227 comments left. 


In [32]:
# Transform CSV to Word File (Supplementary Posts)
# This is done after manual selection
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

df = pd.read_csv("Supplementary Meditation Posts Cleaned.csv", encoding_errors='ignore')
df['text_len'] = df['selftext'].str.len()
df = df.sort_values('text_len', ascending=False)
df = df.drop_duplicates(subset='selftext', keep='first')
df = df.drop(columns='text_len')
df = df.sort_values('title', key=lambda x: x.str.lower())
df = df.reset_index(drop=True)
df.to_csv("Supplementary_Posts_Meditation_Title.csv", index=False, encoding="utf-8")
print(f"Total {len(df)} records. ")

doc = Document()
title = doc.add_heading('Aphantasia & Meditation – Supplementary Posts for Analysis', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER
for i, row in df.iterrows():
    heading = doc.add_paragraph()
    run = heading.add_run(f"{i+1}. {row['title']}")
    run.bold = True
    run.font.size = Pt(14)
    meta = doc.add_paragraph()
    meta_run = meta.add_run(
        f"Subreddit: {row['subreddit']} | Score: {row['score']} | Comments: {row['num_comments']}"
    )
    meta_run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)
    meta_run.font.size = Pt(9)
    doc.add_paragraph(str(row['selftext']))
    doc.add_paragraph("")
doc.save("Supplementary_Posts_Meditation.docx")
print(f"Done! {len(df)} posts saved. ")

Total 16 records. 
Done! 16 posts saved. 


In [1]:
# Comments Filtering (Supplementary Posts)
import pandas as pd
import hashlib

def hash_id(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

posts = pd.read_csv("Supplementary_Posts_Meditation_Title.csv")
posts["id_hash"] = posts["id"].apply(hash_id)
comments = pd.read_csv("Supplementary Meditation Comments Cleaned.csv")
comments['link_id_clean'] = comments['link_id'].str.replace('t3_', '', regex=False)
filtered_comments = comments[comments['link_id_clean'].isin(posts['id_hash'])]
filtered_comments['text_len'] = filtered_comments['body'].str.len()
filtered_comments = filtered_comments.sort_values('text_len', ascending=False)
filtered_comments = filtered_comments.drop_duplicates(subset='body', keep='first')
filtered_comments = filtered_comments.drop(columns=['text_len', 'link_id_clean'])
# Remove Unsupported Posts
filtered_comments = filtered_comments[filtered_comments['score'] > 0]
filtered_comments.to_csv("Supplementary_Comments_Meditation_Title.csv", index=False, encoding="utf-8")
print(f"Total {len(filtered_comments)} comments. ")

Total 213 comments. 


In [6]:
# Supplementary Posts
# Transform Comments CSV to Word File (Group by Posts)
import pandas as pd
import hashlib
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

def hash_id(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

posts = pd.read_csv("Supplementary_Posts_Meditation_Title.csv")
comments = pd.read_csv("Supplementary_Comments_Meditation_Title.csv")
posts["id_hash"] = posts["id"].apply(hash_id)
comments['link_id_clean'] = comments['link_id'].str.replace('t3_', '', regex=False)
doc = Document()
title = doc.add_heading('Aphantasia & Meditation – Supplementary Comments for Analysis', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

total_comments = 0
for i, prow in posts.iterrows():
    post_comments = comments[comments["link_id_clean"] == prow["id_hash"]]
    if len(post_comments) == 0:
        continue
    post_comments = post_comments.sort_values('score', ascending=False)
    heading = doc.add_paragraph()
    run = heading.add_run(f"{i+1}. {prow['title']}")
    run.bold = True
    run.font.size = Pt(14)
    for j, crow in enumerate(post_comments.itertuples(index=False), start=1):
        c_para = doc.add_paragraph()
        c_run = c_para.add_run(f"  [{j}] Score: {crow.score} ")
        c_run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)
        c_run.font.size = Pt(9)
        doc.add_paragraph(str(crow.body))
        total_comments += 1
    doc.add_paragraph("")
doc.save("Supplementary_Comments_Meditation.docx")
print(f"Done! {total_comments} comments saved under {len(posts)} posts. ")

Done! 213 comments saved under 16 posts. 


In [26]:
# Newly Added Posts Scraping
import requests
import csv
import time
import pandas as pd

def download_posts(subreddit, title_filter=None, selftext_filter=None, output_file=None):
    base_url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    all_posts = []
    before = "2026-06-01T00:00:00"
    print(f"Downloading r/{subreddit}...")
    while True:
        params = {
            "subreddit": subreddit,
            "before": before,
            "after": "2023-06-01T00:00:00",
            "limit": 100,
            "sort": "desc"
        }
        if title_filter:
            params["title"] = title_filter
        if selftext_filter:
            params["selftext"] = selftext_filter
        response = requests.get(base_url, params=params)
        data = response.json().get("data") or []
        if not data:
            print("No more data. ")
            break
        all_posts.extend(data)
        print(f"{len(all_posts)} posts downloaded. ")
        before = data[-1].get("created_utc") or data[-1].get("created")
        if len(data) < 100:
            break
        time.sleep(1)
    print(f"Total posts found: {len(all_posts)}")
    if all_posts and output_file:
        all_keys = set()
        for post in all_posts:
            all_keys.update(post.keys())
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=all_keys, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(all_posts)
        print(f"Saved to {output_file}")
download_posts("Meditation", selftext_filter="aphantasia", output_file="Newly Added Meditation Posts.csv")
existing = pd.read_csv("Meditation Posts.csv", low_memory=False)
new = pd.read_csv("Newly Added Meditation Posts.csv", low_memory=False)
existing_ids = set(existing["id"].astype(str))
new = new[~new["id"].astype(str).isin(existing_ids)]
new.to_csv("Newly Added Meditation Posts.csv", index=False, encoding="utf-8")
print(f"New selftext-only posts: {len(new)}")

56 posts downloaded. 
Total posts found: 56
Saved to Newly Added Meditation Posts.csv
New selftext-only posts: 42


In [19]:
# Newly Added Posts Cleaning
import pandas as pd
import re

def clean_posts(filepath):
    df = pd.read_csv(filepath, low_memory=False)
    keep_cols = ['id', 'title', 'selftext', 'author', 'score', 'num_comments', 'subreddit']
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]
    df['selftext'] = df['selftext'].astype(str)
    df['selftext'] = df['selftext'].apply(lambda x: re.sub(r'\n+', ' ', x).strip())
    df = df[~df['selftext'].isin(['[deleted]', '[removed]', 'nan', ''])]
    df = df[df['selftext'].str.strip() != '']
    return df.reset_index(drop=True)

input_files = ["Newly Added Meditation Posts.csv"]

all_dfs = []
for f in input_files:
    cleaned = clean_posts(f)
    print(f"{f}: {len(cleaned)} posts left. ")
    all_dfs.append(cleaned)
combined = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='id')
combined.to_csv("Newly Added Meditation Posts Cleaned.csv", index=False, encoding="utf-8")
print(f"Finished. {len(combined)} posts in total. ")

Newly Added Meditation Posts.csv: 42 posts left. 
Finished. 42 posts in total. 


In [19]:
# Newly Added Comments Scraping (Based on post_id)
import requests
import csv
import time
import pandas as pd
import threading

def flatten_comments(data, result=None):
    if result is None:
        result = []
    for item in data:
        if isinstance(item, dict):
            kind = item.get('kind')
            if kind == 't1':
                comment_data = item.get('data', {})
                result.append({
                    'id': comment_data.get('id'),
                    'body': comment_data.get('body'),
                    'author': comment_data.get('author'),
                    'score': comment_data.get('score'),
                    'link_id': comment_data.get('link_id'),
                })
                replies = comment_data.get('replies')
                if isinstance(replies, dict):
                    children = replies.get('data', {}).get('children', [])
                    flatten_comments(children, result)
    return result

def download_comments_by_post_id(post_id, timeout=4, retries=1):
    for attempt in range(retries):
        result = []
        def fetch():
            try:
                response = requests.get(
                    "https://arctic-shift.photon-reddit.com/api/comments/tree",
                    params={"link_id": f"t3_{post_id}", "limit": 9999},
                    timeout=3
                )
                data = response.json().get("data") or []
                result.extend(flatten_comments(data))
            except:
                pass
        t = threading.Thread(target=fetch, daemon=True)
        t.start()
        t.join(timeout=timeout)
        if not t.is_alive():
            return result
        print(f"Retry {attempt+1} on {post_id}")
    print(f"Skipping {post_id}")
    return []

def download_all_comments(posts_csv, output_file="Newly Added Meditation Comments.csv"):
    df = pd.read_csv(posts_csv)
    post_ids = df['id'].tolist()
    all_comments = []
    print(f"Start downloading comments for {len(post_ids)} posts. ")
    for i, post_id in enumerate(post_ids):
        comments = download_comments_by_post_id(post_id)
        all_comments.extend(comments)
        print(f"[{i+1}/{len(post_ids)}] Post {post_id}: {len(comments)} comments, total {len(all_comments)}")
        time.sleep(0.1)
    if all_comments:
        fieldnames = ['id', 'body', 'author', 'score', 'link_id']
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(all_comments)
        print(f"Finished. {len(all_comments)} comments saved to {output_file}. ")

if __name__ == "__main__":
    download_all_comments("Newly Added Meditation Posts Cleaned.csv")

Start downloading comments for 42 posts. 
[1/42] Post 1tco7zc: 4 comments, total 4
[2/42] Post 1syd3sj: 38 comments, total 42
[3/42] Post 1ogc5qe: 26 comments, total 68
[4/42] Post 1o0ot0p: 27 comments, total 95
[5/42] Post 1nv8iqh: 11 comments, total 106
[6/42] Post 1mefi26: 95 comments, total 201
[7/42] Post 1lcrhkb: 25 comments, total 226
[8/42] Post 1keqeah: 2 comments, total 228
[9/42] Post 1jla1x4: 13 comments, total 241
[10/42] Post 1j4bl0g: 8 comments, total 249
[11/42] Post 1ikpr9t: 13 comments, total 262
[12/42] Post 1ieprt9: 19 comments, total 281
[13/42] Post 1i9l4vr: 18 comments, total 299
[14/42] Post 1hqjo8l: 26 comments, total 325
[15/42] Post 1hcid8e: 4 comments, total 329
[16/42] Post 1h924k3: 19 comments, total 348
[17/42] Post 1gzi9u0: 60 comments, total 408
[18/42] Post 1gvgatj: 2 comments, total 410
[19/42] Post 1gqnjri: 4 comments, total 414
[20/42] Post 1gejw7t: 9 comments, total 423
[21/42] Post 1fs6po8: 0 comments, total 423
[22/42] Post 1fojaay: 14 comments, 

In [20]:
# Newly Added Posts and Comments Hashed
import pandas as pd
import hashlib

def hash_id(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

# Posts
posts = pd.read_csv("Newly Added Meditation Posts Cleaned.csv")
posts['id'] = posts['id'].apply(hash_id)
posts['author'] = posts['author'].apply(hash_id)
posts.to_csv("Newly Added Meditation Posts Hashed.csv", index=False, encoding="utf-8")
print(f"Posts done: {len(posts)}")

# Comments
comments = pd.read_csv("Newly Added Meditation Comments.csv", lineterminator='\n')
comments.columns = comments.columns.str.strip()
comments['id'] = comments['id'].apply(hash_id)
comments['author'] = comments['author'].apply(hash_id)
comments['link_id'] = comments['link_id'].str.replace('t3_', '', regex=False)
comments['link_id'] = comments['link_id'].apply(hash_id)
comments['body'] = comments['body'].apply(lambda x: str(x).replace('\n', ' ').replace('\r', ' '))
comments.to_csv("Newly Added Meditation Comments Hashed.csv", index=False, encoding="utf-8")
print(f"Comments done: {len(comments)}")

Posts done: 42
Comments done: 640


In [21]:
# Newly Added Comments Cleaning
import pandas as pd
import re

df = pd.read_csv("Newly Added Meditation Comments Hashed.csv", lineterminator='\n')
df.columns = df.columns.str.strip()

df['body'] = df['body'].astype(str)
df['body'] = df['body'].apply(lambda x: re.sub(r'\n+', ' ', x).strip())
df = df[~df['body'].isin(['[deleted]', '[removed]', 'nan', ''])]
df = df[df['body'].str.strip() != '']
df = df[df['body'].str.len() > 20]
keep_cols = ['id', 'body', 'author', 'score', 'link_id']
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]
df = df.reset_index(drop=True)
df.to_csv("Newly Added Meditation Comments Cleaned.csv", index=False, encoding="utf-8")
print(f"Finished. {len(df)} comments left. ")

Finished. 603 comments left. 


In [24]:
# Transform CSV to Word File (Newly Added Posts)
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

df = pd.read_csv("Newly Added Meditation Posts Hashed.csv")
df['text_len'] = df['selftext'].str.len()
df = df.sort_values('text_len', ascending=False)
df = df.drop_duplicates(subset='selftext', keep='first')
df = df.drop(columns='text_len')
df = df.sort_values('title', key=lambda x: x.str.lower())
df = df.reset_index(drop=True)
df.to_csv("Newly_Added_Posts_Meditation_Title.csv", index=False, encoding="utf-8")
print(f"Total {len(df)} records. ")

df = pd.read_csv("Newly_Added_Posts_Meditation_Title.csv")
doc = Document()
title = doc.add_heading('Aphantasia & Meditation – Newly Added Posts for Analysis', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

for i, row in df.iterrows():
    heading = doc.add_paragraph()
    run = heading.add_run(f"{i+1}. {row['title']}")
    run.bold = True
    run.font.size = Pt(14)
    meta = doc.add_paragraph()
    meta_run = meta.add_run(
        f"Subreddit: {row['subreddit']} | Score: {row['score']} | Comments: {row['num_comments']}"
    )
    meta_run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)
    meta_run.font.size = Pt(9)
    doc.add_paragraph(str(row['selftext']))
    doc.add_paragraph("")
doc.save("Newly_Added_Posts_Meditation.docx")
print(f"Done! {len(df)} posts saved. ")

Total 42 records. 
Done! 42 posts saved. 


In [22]:
# Newly Added Comments Filtering
import pandas as pd

posts = pd.read_csv("Newly_Added_Posts_Meditation_Title.csv")
comments = pd.read_csv("Newly Added Meditation Comments Cleaned.csv")
comments['link_id_clean'] = comments['link_id'].str.replace('t3_', '', regex=False)
filtered_comments = comments[comments['link_id_clean'].isin(posts['id'])]
filtered_comments['text_len'] = filtered_comments['body'].str.len()
filtered_comments = filtered_comments.sort_values('text_len', ascending=False)
filtered_comments = filtered_comments.drop_duplicates(subset='body', keep='first')
filtered_comments = filtered_comments.drop(columns=['text_len', 'link_id_clean'])
# Remove Unsupported Posts
filtered_comments = filtered_comments[filtered_comments['score'] > 0]
filtered_comments.to_csv("Newly_Added_Comments_Meditation_Title.csv", index=False, encoding="utf-8")
print(f"Total {len(filtered_comments)} comments. ")

Total 577 comments. 


In [23]:
# Newly Added Posts
# Transform Comments CSV to Word File (Group by Posts)
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

posts = pd.read_csv("Newly_Added_Posts_Meditation_Title.csv")
comments = pd.read_csv("Newly_Added_Comments_Meditation_Title.csv")
comments['link_id_clean'] = comments['link_id'].str.replace('t3_', '', regex=False)
doc = Document()
title = doc.add_heading('Aphantasia & Meditation – Newly Added Comments for Analysis', 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

total_comments = 0
for i, prow in posts.iterrows():
    post_comments = comments[comments['link_id_clean'] == prow['id']]
    if len(post_comments) == 0:
        continue
    post_comments = post_comments.sort_values('score', ascending=False)
    heading = doc.add_paragraph()
    run = heading.add_run(f"{i+1}. {prow['title']}")
    run.bold = True
    run.font.size = Pt(14)
    for j, crow in enumerate(post_comments.itertuples(index=False), start=1):
        c_para = doc.add_paragraph()
        c_run = c_para.add_run(f"  [{j}] Score: {crow.score} ")
        c_run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)
        c_run.font.size = Pt(9)
        doc.add_paragraph(str(crow.body))
        total_comments += 1
    doc.add_paragraph("")
doc.save("Newly_Added_Comments_Meditation.docx")
print(f"Done! {total_comments} comments saved under {len(posts)} posts. ")

Done! 577 comments saved under 42 posts. 
